## MobileNet 개요
- 개념: 2017년 구글(Andrew G. Howard 등)이 발표한 경량화 딥러닝 모델 아키텍처
- 목적: 자원이 제한된 모바일·임베디드 기기에서 실시간 추론이 가능하도록 연산량(FLOPs)과 파라미터 수 최소화

## 핵심 메커니즘: Depthwise Separable Convolution
- 2단계 분리 연산 구조
  - 1단계 (Depthwise Conv): 3x3 커널로 각 입력 채널별 공간(Spatial) 정보만 독립 추출 (채널간 섞임 없음)
  - 2단계 (Pointwise Conv): 1x1 커널로 채널(Cross-Channel) 정보를 결합하여 출력 채널 수 조절
- 연산량 절감 효과
  - 연산비율: 1/D_out + 1/K^2 (K: 커널 크기, D_out: 출력 채널 수)
  - 절감 수치: 3x3 커널 기준, 일반 Convolution 대비 연산량·파라미터 약 1/9 수준으로 감소 (80~90% 감축)
- 핵심 이점: 높은 정확도를 유지하면서 컴퓨팅 파워가 부족한 온디바이스(On-Device) 환경에서 실시간 동작 가능

<img src="https://blog.kakaocdn.net/dna/bGRRYD/btszLusiETk/AAAAAAAAAAAAAAAAAAAAAAQAxf_Fw_6AH1dFS48itdbZGNyxcN3uT-AydADUrNVR/img.png?credential=yqXZFxpELC7KVnFOS48ylbz2pIh7yKj8&expires=1793458799&allow_ip=&allow_referer=&signature=On3xCEHP3qAjA9QT8ntgDWioAlg%3D" />

In [1]:
import os
import time
import platform
from shutil import rmtree
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as image
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from torchvision import datasets
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

# 1. 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':  # Mac OS
    plt.rc('font', family='AppleGothic')
else:  # Linux (Colab 등)
    plt.rc('font', family='NanumBarunGothic')

# 2. 마이너스 기호(-) 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

load_dotenv()
# 오류 발생시
# sklearn.set_config(display="text")

True

# Food-11 데이터셋
- 빵, 디저트, 면류, 육류 등 일상에서 흔히 접하는 11가지 주요 음식 카테고리로 구성된 표준 이미지 분류 데이터셋입니다.
다양한 조리 방식과 재료 특성으로 인해 음식 간 시각적 유사성이 높아, 모델의 세밀한 특징 추출 및 다중 클래스 분류 능력을 검증하기에 최적입니다.
복잡한 시각적 패턴을 가진 도메인에서 전이학습(Feature Extraction) 파이프라인 구현과 오분류 분석 역량을 평가하는 데 효과적으로 활용될 수 있습니다.

[food11 데이터셋 링크](https://www.kaggle.com/datasets/trolukovich/food11-image-dataset)  
[food101 데이터셋 링크](https://www.kaggle.com/datasets/giomuntor/food-101-train-val-test)

(예시 : 음식 칼로리 계산 앱 등등 사용 )

In [2]:
# 


In [3]:
base_path = r"/Users/kdt_0900_lyn/ml/resoce/dataset"
food_path = os.path.join(base_path, "food11")

In [4]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

os.makedirs(base_path, exist_ok=True)
# https://www.kaggle.com/datasets/trolukovich/food11-image-dataset
api.dataset_download_files(r"trolukovich/food11-image-dataset", path=base_path, unzip=True)

print("다운로드 완료")

Dataset URL: https://www.kaggle.com/datasets/trolukovich/food11-image-dataset
다운로드 완료


In [5]:
# # 네. 이 값은 ImageNet으로 사전학습된 MobileNet 계열 모델을 쓸 때 사용 (해당 가중치(weights)가 요구하는 전처리)
# mean = [0.485, 0.456, 0.406]
# std = [0.229, 0.224, 0.225]

- [MobileNet Weight](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.mobilenet_v2.html#torchvision.models.mobilenet_v2)


In [9]:
from pathlib import Path

# Kaggle 압축 해제 폴더명 차이를 자동으로 찾아 데이터셋 루트를 설정합니다.
candidates = [
    p for p in Path(base_path).rglob("training")
    if p.is_dir() and any(child.is_dir() for child in p.iterdir())
]
if not candidates:
    raise FileNotFoundError(f"클래스 폴더를 포함한 training 디렉터리를 찾지 못했습니다: {base_path}")

dataset_root = candidates[0].parent
train_path = dataset_root / "training"
val_path = dataset_root / "validation"
test_path = dataset_root / "evaluation"
print(f"데이터셋 경로: {dataset_root}")

데이터셋 경로: /Users/kdt_0900_lyn/ml/resoce/dataset


In [12]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),  # 크기 224*224
    transforms.RandomHorizontalFlip(),  # 좌우 반전 
    transforms.RandomRotation(15),      # 최대 15도로 무작위 회전 
    transforms.ToTensor(),              # 이미지를 텐서로 바꾸기  (0~255 / )
    transforms.Normalize(mean=mean, std=std),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
    
])

In [13]:
train_dataset = ImageFolder(train_path, transform=train_transform)
val_dataset = ImageFolder(val_path, transform=test_transform)  # 검증용 
test_dataset = ImageFolder(test_path, transform=test_transform)

In [14]:
print(len(train_dataset), len(val_dataset), len(test_dataset))

9866 3430 3347


In [15]:
# num_workers=2(CPU코어개수)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=32, num_workers=2)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=32, num_workers=2)
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=32, num_workers=2)

In [16]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
elif torch.xpu.is_available():
    device = torch.device("xpu")
else:
    device = torch.device("cpu")

print(f"현재 선택된 장치: {device}")

현재 선택된 장치: mps


In [17]:
model1 = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

In [18]:
models

<module 'torchvision.models' from '/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torchvision/models/__init__.py'>

## 모델 프리징 


In [22]:
# MobileNet의 모든 파라미터를 고정하여 사전학습 가중치를 보존합니다.(모델 프리징 )
for param in model1.parameters():
    param.requires_grad = False

In [23]:
model1.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=False),
    nn.Linear(model1.classifier[1].in_features, 256),
    nn.BatchNorm1d(256), #가중치 안정화(선택사항)
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 11)
)

In [41]:
model1 = model1.to(device)

In [25]:
model1

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
  

In [34]:
# 분류기 헤드만 학습하도록 손실 함수와 옵티마이저를 설정합니다.
criterion = nn.CrossEntropyLoss()
optimizer1 = optim.Adam(model1.classifier.parameters(), lr=0.001)

In [27]:
def train_one_epoch_multiclass(model, dataloader, criterion, optimizer, device="cpu"):
    model.train()
    sum_losses = 0.0
    sum_accs = 0.0

    for i, (X_batch, y_batch) in enumerate(dataloader, 1):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        batch_loss = 0.0
        
        # 학습
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()

        batch_loss += loss.item()
        sum_losses += batch_loss

        # 다중 클래스의 정확도 계산
        y_pred_index = y_pred.argmax(dim=1)
        acc = (y_batch == y_pred_index).float().mean().item() * 100
        sum_accs += acc

        if i % 20 == 0:
            print(f"[Batch {i}/{len(dataloader)}] Batch Loss: {batch_loss:.4f}, Batch Accuracy: {acc:.2f}%")

    avg_loss = sum_losses / len(dataloader)
    avg_acc = sum_accs / len(dataloader)

    return avg_loss, avg_acc

In [28]:
@torch.no_grad()
def evaluate_multiclass(model, dataloader, criterion, device="cpu"):
    model.eval()
    sum_losses = 0.0
    sum_accs = 0.0

    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        sum_losses += loss.item()

        # 다중 클래스 정확도 계산
        y_pred_index = y_pred.argmax(dim=1)
        acc = (y_batch == y_pred_index).float().mean().item() * 100
        sum_accs += acc

    avg_loss = sum_losses / len(dataloader)
    avg_acc = sum_accs / len(dataloader)
    
    return avg_loss, avg_acc

In [29]:
def train_model_multiclass(model, model_name, train_loader, val_loader, criterion, optimizer, epochs, device, save_base_dir):
    save_dir = os.path.join(save_base_dir, model_name)
    os.makedirs(save_dir, exist_ok=True)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0

    # 학습
    print(f"========================={model_name} 학습 시작 ============================")
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch_multiclass(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate_multiclass(model, val_loader, criterion, device)

        print(f"[Epoch: {epoch + 1}] | Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.2f}%")
        print(f"[Epoch: {epoch + 1}] | Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.2f}%")

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(save_dir, "best_model.pth"))

    torch.save(model.state_dict(), os.path.join(save_dir, f"last_model_{epochs}.pth"))
    print(f"========================={model_name} 학습 종료 ============================")
    return history

In [ ]:
# 모델과 입력 텐서가 같은 장치(MPS/CPU)에 있도록 맞춥니다.
model1 = model1.to(device)

history1 = train_model_multiclass(
    model=model1,
    model_name="Mobilenet-multiclass",
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    criterion=criterion,
    optimizer=optimizer1,
    epochs=10,
    device=device,
    save_base_dir=os.path.join(base_path, "weights")
)

## 모델 평가 

In [ ]:
best_model_path = os.path.join(
    base_path, "weights", "Mobilenet-multiclass", "best_model.pth"
)

In [ ]:
model1.load_state_dict(torch.load(best_model_path, map_location=device))
model1 = model1.to(device)

In [ ]:
model1.eval()

test_loss, test_acc = evaluate_multiclass(
    model=model1,
    dataloader=test_dataloader,
    criterion=criterion,
    device=device
)
print("최종 모델 평가")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

## 예측 시각화 

In [40]:
img, labels = next(iter(test_dataloader))

In [ ]:
calss_to _idx = train_dataset.class_to_idx
class_dict = dict(enumerate(train_dataset.classes)

In [ ]:
imgs, labels = next(iter(test_dataloader))
model1.eval()
imgs_device = imgs.to(device)

with torch.no_grad():
    preds = model1(imgs_device)
    pred_labels = preds.argmax(dim=1).cpu()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
class_names = train_dataset.classes

mean_arr = np.array(mean)
std_arr = np.array(std)

for ax, img, label, pred in zip(axes.flatten(), imgs, labels, pred_labels):
    img_np = img.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np * std_arr + mean_arr, 0, 1)
    true_label = int(label.item())
    pred_label = int(pred.item())
    color = "green" if true_label == pred_label else "red"
    ax.imshow(img_np)
    ax.set_title(f"True: {class_names[true_label]}\nPred: {class_names[pred_label]}", color=color)
    ax.axis("off")

plt.show()

In [ ]:

model1.eval()
imgs_device = imgs.to(device)

with torch.no_grad():
    preds = model1(imgs_device)
    pred_labels = preds.argmax(dim=1).cpu()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))

mean = np.array(mean)
std = np.array(std)

for ax, img, label, pred in zip(axes.flatten(), imgs, labels, pred_labels):
    img_np = img.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np * std + mean, 0, 1)
    true_label = int(label.item())
    pred_label = int(pred.item())
    color = "green" if true_label == pred_label else "red"
    ax.imshow(img_np)
    ax.set_title(f"True: {class_dict[true_label]}\nPred: {class_dict[pred_label]}", color=color)
    ax.axis("off")

plt.show()

In [ ]:
model1.eval()
imgs_device = imgs.to(device)

with torch.no_grad():
    preds = model1(imgs_device)
    pred_labels = preds.argmax(dim=1).cpu()

incorrect_samples = []
incorrect_mask = (pred_labels != labels)

if incorrect_mask.any():
    for img, label, pred in zip(imgs[incorrect_mask], labels[incorrect_mask], pred_labels[incorrect_mask]):
        incorrect_samples.append({
            "img": img,
            "true_label": label.item(),
            "pred_label": pred.item()
        })

print(incorrect_samples)

In [43]:
utk_path = os.path.join(base_path, "utkface")
os.makedirs(utk_path, exist_ok=True)

api = KaggleApi()
api.authenticate()
api.dataset_download_files(r"jangedoo/utkface-new", path=utk_path, unzip=True)

print("다운로드 완료")

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
다운로드 완료


In [46]:
import glob 
utk_img_path = os.path.join(utk_path, r"UTKFace")

all_file = glob.glob(os.path.join(utk_img_path, "*.jpg"))
print(len(all_file))

23708


In [52]:
train_datas, temp_datas = train_test_split(all_file, test_size=0.2, random_state=42)
val_datas, test_datas = train_test_split(temp_datas, test_size=0.5, random_state=42)

print(len(train_datas), len(val_datas), len(test_datas))

18966 2371 2371


In [58]:
class UTKFaceDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        image = Image.open(img_path).convert("RGB")

        file_name = os.path.basename(img_path)
        age = float(file_name.split("_")[0])

        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor([age], dtype=torch.float32)

In [59]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
    
])

In [60]:
train_dataset = UTKFaceDataset(train_datas, transform=train_transform)
val_dataset = UTKFaceDataset(val_datas, transform=train_transform)
test_dataset = UTKFaceDataset(test_datas, transform=train_transform)

In [102]:

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [103]:
imgs, labels = next(iter(train_dataloader))
print(imgs.size(), labels.size())

torch.Size([32, 3, 224, 224]) torch.Size([32, 1])


In [65]:
model2 = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

In [66]:
model2


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
  

In [88]:
# 모델프리징 
for param in model2.parameters():
    param.requires_grad = False

In [89]:
in_teatyres = model2.classifier[1].in_features
model2.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=False),
    nn.Linear(1280, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 1)
)

In [90]:
model2 = model2.to(device)

### AdamW (Adam with Decoupled Weight Decay)
- AdamW는 기존 Adam의 가중치 감쇠(Weight Decay) 연산 방식을 개선하여 모델의 일반화 성능을 높인 최적화 알고리즘입니다.

#### 핵심 특징
- Weight Decay 분리 (Decoupling): 기존 Adam은 가중치 감쇠가 기울기(Gradient) 계산에 얽혀 모멘텀을 왜곡했으나, AdamW는 이를 완전히 분리하여 순수 기울기로만 모멘텀을 계산하고 가중치 감쇠는 별도로 적용합니다.
- 과적합 방지 및 일반화: Adam의 빠른 수렴 속도를 유지하면서, 전이학습(Transfer Learning) 시 사전 학습된 가중치의 과적합을 효과적으로 억제합니다.

#### 주요 하이퍼파라미터
- params: model.parameters()
  최적화할 모델의 가중치 패러미터 집합
- lr: 1e-4 ~ 3e-4
  학습률 (Learning Rate): 매 스텝 가중치 업데이트 크기 (전이학습 시 1e-4 이하 권장)
- weight_decay: 1e-2 (0.01)
  가중치 감쇠: 가중치가 과도하게 커지는 것을 방지하는 정규화(Regularization) 계수
- betas: (0.9, 0.999)
  모멘텀 계수 (beta1, beta2): beta1(1차 기울기 관성), beta2(2차 기울기 제곱 보폭) 지수 이동 평균 비율
- eps: 1e-8
  수치 안정성 상수 (epsilon): 0으로 나누는 연산 오류 방지용 미소값`m

In [91]:
# 절댓값 오차 
criterion = nn.L1Loss()
optimizer2 = optim.Adam(model2.classifier.parameters(), lr=1e-4, weight_decay=1e-2)


In [97]:
def train_one_epoch_regression(model, dataloader, criterion, optimizer, device="cpu"):
    model.train()
    sum_losses = sum_maes = sum_mses = sum_rmses = 0.0

    for i, (X_batch, y_batch) in enumerate(dataloader, 1):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).float().view(-1, 1)

        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()

        preds_flat = y_pred.detach().squeeze()
        targets_flat = y_batch.squeeze()
        mae = torch.abs(preds_flat - targets_flat).mean().item()
        mse = ((preds_flat - targets_flat) ** 2).mean().item()
        rmse = mse ** 0.5

        sum_losses += loss.item()
        sum_maes += mae
        sum_mses += mse
        sum_rmses += rmse

        if i % 20 == 0:
            print(f"[Batch {i}/{len(dataloader)}] Loss: {loss.item():.4f}, MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}")

    length = len(dataloader)
    return sum_losses / length, sum_maes / length, sum_mses / length, sum_rmses / length

In [98]:
@torch.no_grad()
def evaluate_regression(model, dataloader, criterion, device="cpu"):
    model.eval()
    sum_losses = sum_maes = sum_mses = sum_rmses = 0.0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).float().view(-1, 1)

        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)

        preds_flat = y_pred.squeeze()
        targets_flat = y_batch.squeeze()
        mae = torch.abs(preds_flat - targets_flat).mean().item()
        mse = ((preds_flat - targets_flat) ** 2).mean().item()
        rmse = mse ** 0.5

        sum_losses += loss.item()
        sum_maes += mae
        sum_mses += mse
        sum_rmses += rmse

    length = len(dataloader)
    return sum_losses / length, sum_maes / length, sum_mses / length, sum_rmses / length

In [100]:
def train_model_regression(model, model_name, train_loader, val_loader, criterion, optimizer, epochs, device, save_base_dir):
    model = model.to(device)
    save_dir = os.path.join(save_base_dir, model_name)
    os.makedirs(save_dir, exist_ok=True)

    history = {
        "train_loss": [], "val_loss": [],
        "train_mae": [], "val_mae": [], 
        "train_mse": [], "val_mse": [], 
        "train_rmse": [], "val_rmse": []
              
    }
    best_val_loss = float("inf")

    print(f"{'=' * 25}{model_name} 학습 시작{'=' * 25}")
    for epoch in range(epochs):
        train_loss, train_mae, train_mse, train_rmse = train_one_epoch_regression(model, train_loader, criterion, optimizer, device)
        val_loss, val_mae, val_mse, val_rmse = evaluate_regression(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_mae"].append(train_mae)
        history["val_mae"].append(val_mae)
        history["train_mse"].append(train_mse)
        history["val_mse"].append(val_mse)
        history["train_rmse"].append(train_rmse)
        history["val_rmse"].append(val_rmse)

        print(f"[Epoch {epoch + 1}/{epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"                 Train MAE: {train_mae:.4f} | Val MAE: {val_mae:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(save_dir, "best_model.pth"))

    torch.save(model.state_dict(), os.path.join(save_dir, f"last_model_{epochs}.pth"))
    print(f"{'=' * 25}{model_name} 학습 종료{'=' * 25}")
    return history

In [104]:
history2 = train_model_regression(
    model=model2,
    model_name="Mobilenet-regression",
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    criterion=criterion,
    optimizer=optimizer2,
    epochs=10,
    device=device,
    
    
    save_base_dir=os.path.join(base_path, "weights")
)

=========================Mobilenet-regression 학습 시작=========================
[Batch 20/593] Loss: 27.4942, MAE: 27.4942, MSE: 984.0063, RMSE: 31.3689
[Batch 40/593] Loss: 37.3642, MAE: 37.3642, MSE: 1726.8043, RMSE: 41.5548
[Batch 60/593] Loss: 28.3302, MAE: 28.3302, MSE: 1230.4402, RMSE: 35.0776
[Batch 80/593] Loss: 29.2411, MAE: 29.2411, MSE: 1276.6045, RMSE: 35.7296
[Batch 100/593] Loss: 31.2780, MAE: 31.2780, MSE: 1335.8654, RMSE: 36.5495
[Batch 120/593] Loss: 19.4119, MAE: 19.4119, MSE: 552.4202, RMSE: 23.5036
[Batch 140/593] Loss: 22.8208, MAE: 22.8208, MSE: 871.3189, RMSE: 29.5181
[Batch 160/593] Loss: 18.2218, MAE: 18.2218, MSE: 481.5716, RMSE: 21.9447
[Batch 180/593] Loss: 18.9005, MAE: 18.9005, MSE: 547.4819, RMSE: 23.3983
[Batch 200/593] Loss: 20.8589, MAE: 20.8589, MSE: 780.5696, RMSE: 27.9387
[Batch 220/593] Loss: 16.9692, MAE: 16.9692, MSE: 598.6154, RMSE: 24.4666
[Batch 240/593] Loss: 20.5080, MAE: 20.5080, MSE: 640.4796, RMSE: 25.3077
[Batch 260/593] Loss: 18.1213, MAE:

In [107]:
model_name = "Mobilenet-regression"
best_model_path = os.path.join(base_path, "weights", model_name, "best_model.pth")

# 가중치 없이 base_model만 가져오기
model3 = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# 모델 프리징
for param in model2.parameters():
    param.requires_grad = False

in_features = model3.classifier[1].in_features
model3.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=False),
    nn.Linear(1280, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 1)
)

model3.load_state_dict(torch.load(best_model_path))
model3 = model3.to(device)

In [108]:
img_path = r""  # 이미지 파일 경로
img = Image.open(img_path).convert("RGB")
img_tensor = test_transform(img)
img_batch = img_tensor.unsqueeze(0).to(device)

model3.eval()
with torch.no_grad():
    output = model3(img_batch)
    predicted_age = output.item()

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.axis("off")
plt.set_title(f"실 나이: {}살 | 예측 나이: {predicted_age}살")
plt.show()

SyntaxError: f-string: valid expression required before '}' (357497537.py, line 14)